# Supporting Information
## Computation of Wavefunction Overlap Integrals for Quantum Phase Estimation

**Reference:** Sugisaki, K. *et al.* *ACS Cent. Sci.* **2019**, *5*, 167-175. DOI: 10.1021/acscentsci.8b00788

---

This notebook provides a fully self-contained, reproducible implementation of the method for computing
wavefunction overlap integrals between a CASSCF reference wavefunction and three trial wavefunctions:

| Trial wavefunction | Symbol | Physical meaning |
|---|---|---|
| Restricted Hartree-Fock | RHF | Closed-shell singlet, spin-symmetric |
| Broken-symmetry UHF | BS-UHF | Spin-polarised, spin-contaminated |
| Spin-projected UHF | PUHF | Symmetry-restored open-shell singlet |

The overlap $|\langle\Psi_{\rm CAS}|\Phi_{\rm trial}\rangle|^2$
is the **success probability** of the quantum phase estimation algorithm (QPEA).

### Notebook structure

| Section | Content |
|---|---|
| 1 | Setup, imports, and molecular definition |
| 2 | RHF reference and broken-symmetry UHF |
| 3 | Natural orbital analysis via the Lowdin transformation |
| 4 | Diradical characters (Sugisaki Eq. 4) |
| 5 | CASSCF reference wavefunction |
| 6 | RHF-CASSCF overlap and sanity check |
| 7 | Raw UHF-CASSCF overlap |
| 8 | Spin projection and PUHF normalisation |
| 9 | PUHF-CASSCF overlap (main result) |
| 10 | Summary table and physical interpretation |


---
## Section 1 -- Setup and Molecular Definition


In [ ]:
import numpy as np
from pyscf import gto, scf, mcscf, fci
from pyscf.fci import cistring

# np.linalg.eigh  -- eigendecomposition of real symmetric matrix (LAPACK dsyev)
# np.linalg.det   -- determinant via LU factorisation


In [ ]:
# ---------------------------------------------------------------
# Molecular definition
# ---------------------------------------------------------------
# Replace XYZ_FILE with the path to your geometry file.
# The file must be in standard XYZ format:
#   Line 1: number of atoms
#   Line 2: comment (ignored)
#   Lines 3+: Element  X  Y  Z  (Angstrom)
#
# symmetry=False is required: the broken-symmetry UHF must be
# able to break spatial symmetry freely.

XYZ_FILE  = 'your_molecule.xyz'   # <-- replace with your geometry file
BASIS     = 'def2-svp'
N_CAS_ORB = 8    # number of active orbitals
N_CAS_EL  = 8    # number of active electrons (split 4 alpha + 4 beta)

def read_xyz(filename):
    with open(filename) as f:
        lines = f.readlines()
    nat = int(lines[0])
    return ''.join(lines[2:2 + nat])

mol = gto.M(
    atom     = read_xyz(XYZ_FILE),
    basis    = BASIS,
    spin     = 0,
    charge   = 0,
    symmetry = False,
    verbose  = 3
)

print(f'Molecule:     {XYZ_FILE}')
print(f'Electrons:    {mol.nelectron}  ({mol.nelec[0]} alpha, {mol.nelec[1]} beta)')
print(f'Basis:        {BASIS}  ({mol.nao} AOs)')
print(f'Active space: CAS({N_CAS_EL},{N_CAS_ORB})')


---
## Section 2 -- RHF Reference and Broken-Symmetry UHF

### 2.1 Restricted Hartree-Fock

The RHF wavefunction $|\Phi_{\rm RHF}\rangle$ is a single Slater determinant
in which $\alpha$ and $\beta$ electrons share identical spatial orbitals.
The MO coefficient matrix satisfies the Roothaan-Hall equations:

$$\mathbf{F}\mathbf{C} = \mathbf{S}\mathbf{C}\boldsymbol{\varepsilon},
\qquad \mathbf{C}^\top\mathbf{S}\mathbf{C} = \mathbf{I}.$$

For diradicaloid systems, RHF has low overlap with the exact ground state
because it cannot represent open-shell singlet character arising from
near-degenerate frontier orbitals.

---

### 2.2 Broken-Symmetry UHF Initial Guess

We apply **opposite HOMO-LUMO rotations** to $\alpha$ and $\beta$ spin-orbital sets.
Let $\varphi_H, \varphi_L$ be the RHF HOMO and LUMO.  The rotated orbitals are:

$$\varphi_H^{\,\alpha} = \cos\theta\,\varphi_H + \sin\theta\,\varphi_L,
\qquad \varphi_H^{\,\beta}  = \cos\theta\,\varphi_H - \sin\theta\,\varphi_L,$$

$$\varphi_L^{\,\alpha} = -\sin\theta\,\varphi_H + \cos\theta\,\varphi_L,
\qquad \varphi_L^{\,\beta}  = \sin\theta\,\varphi_H + \cos\theta\,\varphi_L.$$

The **opposite signs** for $\beta$ break spin symmetry: $\alpha$ electrons
localise preferentially on one lobe, $\beta$ on the other,
mimicking the two-determinant open-shell singlet.
At $\theta = \pi/4$ the mixing is maximised (50/50 character).
The rotated density matrices seed a Newton-DIIS UHF convergence.


In [ ]:
# ---------------------------------------------------------------
# Step 1: RHF reference
# ---------------------------------------------------------------
mf_rhf = scf.RHF(mol).run()
E_rhf  = mf_rhf.e_tot

homo_gap = (mf_rhf.mo_energy[mol.nelectron // 2]
            - mf_rhf.mo_energy[mol.nelectron // 2 - 1]) * 27.211

print(f'E(RHF)        = {E_rhf:.10f} Ha')
print(f'HOMO-LUMO gap = {homo_gap:.3f} eV')

# ---------------------------------------------------------------
# Step 2: Broken-symmetry UHF
# ---------------------------------------------------------------
theta = np.pi / 4      # 45-degree mixing maximises HOMO-LUMO overlap
C     = mf_rhf.mo_coeff
nocc  = np.count_nonzero(mf_rhf.mo_occ > 0)
homo, lumo = nocc - 1, nocc

Ca = C.copy()   # alpha MO coefficients
Cb = C.copy()   # beta  MO coefficients

# Alpha: +theta rotation
Ca[:, homo] =  np.cos(theta)*C[:, homo] + np.sin(theta)*C[:, lumo]
Ca[:, lumo] = -np.sin(theta)*C[:, homo] + np.cos(theta)*C[:, lumo]

# Beta: -theta rotation (opposite sign = spin symmetry breaking)
Cb[:, homo] =  np.cos(theta)*C[:, homo] - np.sin(theta)*C[:, lumo]
Cb[:, lumo] =  np.sin(theta)*C[:, homo] + np.cos(theta)*C[:, lumo]

nalpha, nbeta = mol.nelec
dm_a = Ca[:, :nalpha] @ Ca[:, :nalpha].T
dm_b = Cb[:, :nbeta]  @ Cb[:, :nbeta].T

mf_uhf = scf.UHF(mol).newton()
mf_uhf.kernel(dm0=(dm_a, dm_b))

s2 = mf_uhf.spin_square()[0]
print(f'E(UHF)        = {mf_uhf.e_tot:.10f} Ha')
print(f'<S2>(UHF)     = {s2:.6f}  (ideal singlet: 0.000; open-shell singlet: ~1.0)')
if s2 < 0.1:
    print('  WARNING: <S2> near 0 -- UHF may have collapsed to RHF.')


---
## Section 3 -- Natural Orbital Analysis via the Lowdin Transformation

### 3.1 One-Particle Reduced Density Matrix (1-RDM)

The **spin-summed 1-RDM** in the AO basis is:

$$\boldsymbol{\gamma} = \mathbf{P}^\alpha + \mathbf{P}^\beta,
\qquad \mathbf{P}^\sigma = \mathbf{C}^\sigma_{\rm occ}\,(\mathbf{C}^\sigma_{\rm occ})^\top.$$

Fractional eigenvalues of $\boldsymbol{\gamma}$ (in an orthogonal basis)
signal multi-reference (diradical) character.

---

### 3.2 Lowdin Symmetric Orthogonalisation

Diagonalise the AO overlap matrix:

$$\mathbf{S} = \mathbf{U}\,\boldsymbol{\Lambda}\,\mathbf{U}^\top,
\qquad \boldsymbol{\Lambda} = {\rm diag}(\lambda_1,\ldots), \quad \lambda_i > 0.$$

The symmetric square root and its inverse:

$$\mathbf{S}^{1/2}  = \mathbf{U}\,\boldsymbol{\Lambda}^{1/2}\,\mathbf{U}^\top,
\qquad \mathbf{S}^{-1/2} = \mathbf{U}\,\boldsymbol{\Lambda}^{-1/2}\,\mathbf{U}^\top.$$

The orthogonalised 1-RDM:

$$\tilde{\boldsymbol{\gamma}} = \mathbf{S}^{1/2}\,\boldsymbol{\gamma}\,\mathbf{S}^{1/2}.$$

---

### 3.3 Natural Orbitals in the AO Basis

Diagonalise $\tilde{\boldsymbol{\gamma}} = \tilde{\mathbf{C}}\,{\rm diag}(n_i)\,\tilde{\mathbf{C}}^\top$,
then back-transform:

$$\mathbf{C}_{\rm NO} = \mathbf{S}^{-1/2}\,\tilde{\mathbf{C}}.$$

The natural orbital coefficient matrix is **S-orthonormal**:
$\mathbf{C}_{\rm NO}^\top\mathbf{S}\mathbf{C}_{\rm NO} = \mathbf{I}$,
and the 1-RDM reconstructs as
$\boldsymbol{\gamma} = \mathbf{C}_{\rm NO}\,{\rm diag}(n_i)\,\mathbf{C}_{\rm NO}^\top$.
The eigenvalues $\{n_i\}$ are the **natural orbital occupation numbers (NOONs)**,
$n_i \in [0,2]$.

| System | $n_{\rm HONO}$ | $n_{\rm LUNO}$ | Character |
|---|---|---|---|
| Closed-shell singlet | ~2.0 | ~0.0 | single-reference |
| Pure open-shell singlet | ~1.0 | ~1.0 | diradical |
| Intermediate | 1-2 | 0-1 | partial diradical |


In [ ]:
# ---------------------------------------------------------------
# Step 3: UHF Natural Orbitals via Lowdin transformation
# ---------------------------------------------------------------

# 3.1  Spin-summed 1-RDM
dm_a_conv, dm_b_conv = mf_uhf.make_rdm1()   # (nao,nao) each
dm_total = dm_a_conv + dm_b_conv

# 3.2  Lowdin square roots
S_ao    = mol.intor('int1e_ovlp')
eigS, U = np.linalg.eigh(S_ao)
S_half     = U @ np.diag(np.sqrt(eigS))       @ U.T   # S^{+1/2}
S_inv_half = U @ np.diag(1.0/np.sqrt(eigS))   @ U.T   # S^{-1/2}

# 3.3  Orthogonalised 1-RDM -> eigenvectors in Lowdin basis
dm_orth          = S_half @ dm_total @ S_half
occ_raw, C_orth  = np.linalg.eigh(dm_orth)

# Sort descending (doubly occupied first)
idx       = np.argsort(occ_raw)[::-1]
occ       = occ_raw[idx]
C_orth    = C_orth[:, idx]

# 3.4  Back-transform to AO basis
C_no = S_inv_half @ C_orth

# ---------------------------------------------------------------
# Verification: C_no must be S-orthonormal
# ---------------------------------------------------------------
check = C_no.T @ S_ao @ C_no
print(f'S-orthonormality: max|C_NO^T S C_NO - I| = {np.max(np.abs(check - np.eye(mol.nao))):.2e}')
print('  (should be < 1e-10)')

n_occ = mol.nelectron // 2
print('\nFrontier NOONs:')
print(f'  {'Orbital':>8}  {'NOON':>10}  {'Character'}')
print('  ' + '-'*42)
for offset, label in [(-2,'HONO-1'),(-1,'HONO'),(0,'LUNO'),(1,'LUNO+1')]:
    n_i = occ[n_occ + offset]
    char = '<-- diradical' if 0.2 < n_i < 1.8 else ('doubly occ' if n_i > 1.8 else 'virtual')
    print(f'  {label:>8}  {n_i:>10.6f}  {char}')
print(f'\nTotal from NOONs: {occ.sum():.6f}  (should = {mol.nelectron})')


---
## Section 4 -- Diradical Characters

### 4.1 Without Spin Projection (Plain UHF)

The simplest measure of diradical character [Dohnert & Koutecky, 1980] is
the occupation number of the $i$-th LUNO:

$$y_i^{\rm UHF} = n_{{\rm LUNO}+i} \in [0,1].$$

**Limitation:** UHF wavefunctions are spin-contaminated, causing the raw NOONs
to **overestimate** true diradical character.

---

### 4.2 With Spin Projection (Sugisaki 2019, Eq. 4)

The spin-projected diradical character at the PUHF level is:

$$\boxed{
y_i^{\rm PUHF} = \frac{n_{{\rm LUNO}+i} - 2(1 - n_{{\rm LUNO}+i})}
                      {1 + (1 - n_{{\rm LUNO}+i})^2}
}$$

Setting $x = n_{{\rm LUNO}+i}$: $y_i^{\rm PUHF} = (3x-2)/[1+(1-x)^2]$.

**Physical domain:** For $x < 2/3$ the numerator is negative.
Values are clipped: $y_i^{\rm PUHF} \leftarrow \max(0,\,y_i^{\rm PUHF})$.
The formula is physically meaningful only when $n_{{\rm LUNO}+i} > 2/3$.

**Effect of spin projection:** $y_i^{\rm PUHF} \leq y_i^{\rm UHF}$ always --
spin projection removes the triplet contamination, reducing the estimated diradical character.


In [ ]:
# ---------------------------------------------------------------
# Step 4: Diradical characters
# ---------------------------------------------------------------

n_pairs = min(n_occ, len(occ) - n_occ)
y_uhf   = np.zeros(n_pairs)
y_puhf  = np.zeros(n_pairs)

for i in range(n_pairs):
    x = occ[n_occ + i]          # n_{LUNO+i}
    y_uhf[i]  = x
    # Sugisaki Eq. 4
    num       = x - 2.0*(1.0-x)
    denom     = 1.0 + (1.0-x)**2
    y_puhf[i] = float(np.clip(num/denom, 0.0, 1.0))

print(f"{'i':>4}  {'n_LUNO+i':>12}  {'y(UHF)':>10}  {'y(PUHF)':>10}  {'Reduction':>12}")
print('  '+'-'*54)
for i in range(min(4, n_pairs)):
    x = occ[n_occ+i]
    print(f'  {i:>2}  {x:>12.6f}  {y_uhf[i]:>10.6f}  {y_puhf[i]:>10.6f}  {y_uhf[i]-y_puhf[i]:>+12.6f}')
print()
print(f'Threshold n > 2/3 = {2/3:.6f}: PUHF formula meaningful above this.')


---
## Section 5 -- CASSCF Reference Wavefunction

### 5.1 CASSCF as High-Level Reference

$$|\Psi_{\rm CAS}\rangle = \sum_I c_I |D_I\rangle, \qquad \sum_I c_I^2 = 1.$$

Here $I=(I_\alpha,I_\beta)$ labels a pair of alpha/beta occupation strings
over $n_{\rm orb}$ active orbitals.  PySCF stores $c_I$ in `mc.ci[ia, ib]`.

Orbital partition:
- **Core** ($n_{\rm core}$): doubly occupied in all determinants
- **Active** ($n_{\rm orb}$): variable occupation, encoded by CI strings
- **Virtual**: unoccupied in all CI strings

### 5.2 Active-Space Natural Occupations

The CAS 1-RDM diagonal elements $(\boldsymbol{\gamma}^{\rm CAS})_{pq}$
provide the ground-truth reference for comparison with UHF NOONs.


In [ ]:
# ---------------------------------------------------------------
# Step 5: CASSCF reference (reuse mf_rhf from Step 1)
# ---------------------------------------------------------------

mc = mcscf.CASSCF(mf_rhf, N_CAS_ORB, N_CAS_EL)
mc.fcisolver = fci.direct_spin1.FCI(mol)
mc.kernel()

print(f'E(CASSCF)         = {mc.e_tot:.10f} Ha')
print(f'Correlation energy = {(mc.e_tot - E_rhf)*1000:.4f} mHa')
print(f'Core orbitals:    {mc.ncore}')

# Active-space natural occupations
dm1_cas    = mc.fcisolver.make_rdm1(mc.ci, mc.ncas, mc.nelecas)
occ_cas, _ = np.linalg.eigh(dm1_cas)
occ_cas    = np.sort(occ_cas)[::-1]

print('\nCAS active natural occupations:')
for i, o in enumerate(occ_cas):
    char = '<-- diradical' if 0.5 < o < 1.5 else ('doubly occ' if o > 1.5 else 'virtual')
    print(f'  Orb {i:>2}: {o:.6f}  {char}')

# Leading CI configurations
print('\nLeading CI configurations (|c_I|^2 > 0.01):')
na, nb   = mc.nelecas
a_str_all = list(cistring.make_strings(range(mc.ncas), na))
b_str_all = list(cistring.make_strings(range(mc.ncas), nb))
ci_list  = [(mc.ci[ia,ib]**2, mc.ci[ia,ib])
             for ia in range(len(a_str_all))
             for ib in range(len(b_str_all))
             if abs(mc.ci[ia,ib]) > 0.01]
ci_list.sort(reverse=True)
for w,c in ci_list[:5]:
    print(f'  |c|^2 = {w:.5f}   c = {c:+.6f}')


---
## Section 6 -- RHF-CASSCF Overlap and Sanity Check

### Mathematical Derivation

The overlap between the CASSCF wavefunction and a single Slater determinant
$|\Phi\rangle$ is evaluated by expanding in CI configurations
(Sugisaki 2019, Section 2.3):

$$\langle\Psi_{\rm CAS}|\Phi\rangle
= \sum_I c_I \langle D_I|\Phi\rangle
= \sum_I c_I\,
  \det\!\bigl(\mathbf{M}^\alpha[\mathbf{f}^\alpha_I,:]\bigr)\cdot
  \det\!\bigl(\mathbf{M}^\beta[\mathbf{f}^\beta_I,:]\bigr).$$

**Overlap matrices:** For spin $\sigma\in\{\alpha,\beta\}$:

$$\mathbf{M}^\sigma = \mathbf{C}_{\rm CAS}^\top\,\mathbf{S}_{\rm AO}\,
  \mathbf{C}^\sigma_{\Phi,{\rm occ}} \in \mathbb{R}^{N_{\rm MO}^{\rm CAS}\times n_\sigma}.$$

The AO overlap matrix $\mathbf{S}_{\rm AO}$ is included because
the CAS and trial orbital sets are **not in general orthogonal to each other**.
The element $\mathbf{M}^\sigma_{pj} = \langle\phi^{\rm CAS}_p|\psi^\sigma_j\rangle$.

**Row selection:** For CI string $I_\alpha$, the occupied CAS MOs are
the core orbitals $(0\ldots n_{\rm core}-1)$ plus the active orbitals
selected by $I_\alpha$.  Let $\mathbf{f}^\alpha_I$ be this index list of length
$n_\alpha = n_{\rm core} + n_\alpha^{\rm active}$.
Then $\mathbf{M}^\alpha[\mathbf{f}^\alpha_I,:]$ is an $n_\alpha\times n_\alpha$
square matrix and its determinant equals $\langle D_I^\alpha|\Phi^\alpha\rangle$.

**Factorisation:** The overlap factorises into $\alpha$ and $\beta$ parts because
Slater determinants are antisymmetrised products of spin-orbitals and the
$\alpha/\beta$ Fock spaces are orthogonal.

**RHF special case:** $\mathbf{C}^\alpha = \mathbf{C}^\beta = \mathbf{C}^{\rm RHF}$,
so $\mathbf{M}^\alpha = \mathbf{M}^\beta$ and each configuration contributes
$c_I\cdot\det(\cdot)^2$.

### Sanity Check

Passing the **CASSCF orbitals themselves** as the trial wavefunction
($\mathbf{C}^\alpha = \mathbf{C}^\beta = \mathbf{C}_{\rm CAS}$) must return
the **leading CI coefficient $c_0$** (not 1.0):

- $\mathbf{M}^\sigma = \mathbf{C}_{\rm CAS}^\top\mathbf{S}\mathbf{C}_{\rm CAS} = \mathbf{I}$
  (CASSCF orbitals are orthonormal).
- For the reference determinant (core + lowest active orbitals):
  $\det(\mathbf{I}[\mathbf{f}_0,:]) = 1$.
- For all other CI strings: the row submatrix includes rows beyond the first
  $n_{\rm occ}$ columns, giving $\det = 0$.
- Therefore: $\langle\Psi_{\rm CAS}|\mathbf{C}_{\rm CAS}\rangle = c_0$.

This provides a direct numerical verification of the implementation.


In [ ]:
# ---------------------------------------------------------------
# Core function: CASSCF-determinant overlap (Sugisaki expansion)
# ---------------------------------------------------------------

def compute_overlap_sugisaki(mol, Ca, Cb, mc):
    """
    Compute <Psi_CAS | Phi> where |Phi> has alpha MO coeff Ca
    and beta MO coeff Cb.

    Method: Sugisaki et al. 2019, Section 2.3 determinant expansion.
    """
    S_ao  = mol.intor('int1e_ovlp')
    C_cas = mc.mo_coeff
    ncore          = mc.ncore
    ncas           = mc.ncas
    neleca, nelecb = mc.nelecas
    nocc_a = ncore + neleca   # total alpha occupied
    nocc_b = ncore + nelecb   # total beta  occupied

    # Overlap matrices: rows=CAS MOs, cols=trial occupied MOs
    M_a = C_cas.T @ S_ao @ Ca[:, :nocc_a]   # (nmo_cas, nocc_a)
    M_b = C_cas.T @ S_ao @ Cb[:, :nocc_b]   # (nmo_cas, nocc_b)

    a_strings = cistring.make_strings(range(ncas), neleca)
    b_strings = cistring.make_strings(range(ncas), nelecb)
    core_idx  = list(range(ncore))
    total     = 0.0

    for ia, a_str in enumerate(a_strings):
        # bit k=1 in a_str -> active orbital k is occupied
        a_idx      = [i for i in range(ncas) if (a_str & (1 << i))]
        full_a     = core_idx + [i + ncore for i in a_idx]  # CAS MO indices
        sub_a      = M_a[full_a, :]   # (nocc_a, nocc_a) square

        for ib, b_str in enumerate(b_strings):
            c_I = mc.ci[ia, ib]
            if abs(c_I) < 1e-12:
                continue
            b_idx  = [i for i in range(ncas) if (b_str & (1 << i))]
            full_b = core_idx + [i + ncore for i in b_idx]
            sub_b  = M_b[full_b, :]
            total += c_I * np.linalg.det(sub_a) * np.linalg.det(sub_b)

    return total


# ---------------------------------------------------------------
# Step 6a: RHF-CASSCF overlap
# ---------------------------------------------------------------
O_rhf = compute_overlap_sugisaki(mol, mf_rhf.mo_coeff, mf_rhf.mo_coeff, mc)
print('RHF-CASSCF overlap')
print(f'  |<CAS|RHF>|   = {abs(O_rhf):.8f}')
print(f'  |<CAS|RHF>|^2 = {abs(O_rhf)**2:.8f}  (QPEA success probability)')

# ---------------------------------------------------------------
# Step 6b: Sanity check -- pass CASSCF MOs as trial wavefunction
# ---------------------------------------------------------------
# Expected: result should equal the leading CI coefficient c0
O_sanity = compute_overlap_sugisaki(mol, mc.mo_coeff, mc.mo_coeff, mc)
c0 = mc.ci[0, 0]
print(f'\nSanity check: <CAS | CAS MOs>')
print(f'  Computed overlap = {O_sanity:.8f}')
print(f'  Leading CI coeff c0 = {c0:.8f}')
if abs(O_sanity - c0) < 1e-5:
    print('  => PASS: overlap matches c0 as expected.')
else:
    print(f'  => WARNING: difference = {abs(O_sanity-c0):.2e}  (expected < 1e-5)')


---
## Section 7 -- Raw UHF-CASSCF Overlap

The raw UHF wavefunction $|\Phi\rangle$ uses different $\alpha$ and $\beta$
MO sets ($\mathbf{C}^\alpha \neq \mathbf{C}^\beta$).
We compute both $\langle\Psi_{\rm CAS}|\Phi\rangle$ and
$\langle\Psi_{\rm CAS}|\tilde\Phi\rangle$, where $|\tilde\Phi\rangle$ is the
**spin-flipped** partner (swap $\alpha\leftrightarrow\beta$).
These are the two numerator terms of the PUHF overlap formula (Section 9).

**Sign diagnostic:** For a proper broken-symmetry singlet both overlaps should be
positive.  Opposite signs indicate the UHF has not converged to a singlet BS state.


In [ ]:
# ---------------------------------------------------------------
# Step 7: Raw UHF-CASSCF overlaps (numerator terms for PUHF)
# ---------------------------------------------------------------
Ca_uhf, Cb_uhf = mf_uhf.mo_coeff

O_phi       = compute_overlap_sugisaki(mol, Ca_uhf, Cb_uhf, mc)  # original
O_phi_tilde = compute_overlap_sugisaki(mol, Cb_uhf, Ca_uhf, mc)  # alpha<->beta swapped

print('Raw UHF overlaps (before spin projection):')
print(f'  <CAS|Phi>       = {O_phi:+.8f}  (alpha=Ca, beta=Cb)')
print(f'  <CAS|Phi_tilde> = {O_phi_tilde:+.8f}  (alpha=Cb, beta=Ca)')

if (abs(O_phi) > 1e-6 and abs(O_phi_tilde) > 1e-6
        and np.sign(O_phi) != np.sign(O_phi_tilde)):
    print('  WARNING: opposite signs -- UHF may not be a broken-symmetry singlet.')
else:
    print('  Sign check: PASS (both overlaps have the same sign).')


---
## Section 8 -- Singlet Spin Projection and PUHF Normalisation

### 8.1 Why Spin Projection Is Needed

The broken-symmetry UHF wavefunction $|\Phi\rangle$ is not an eigenfunction of
$\hat{S}^2$; it contains triplet contamination:

$$|\Phi\rangle = c_S|S=0\rangle + c_T|S=1\rangle + \cdots$$

Spin projection extracts the $S=0$ component.

### 8.2 Lowdin Singlet Projection (Two-Determinant Approximation)

The projected state is (Lowdin 1955; Yamaguchi 1975):

$$|\Psi_{\rm PUHF}\rangle = \hat{P}_{S=0}|\Phi\rangle
\approx \mathcal{N}\bigl(|\Phi\rangle + |\tilde\Phi\rangle\bigr),$$

where $|\tilde\Phi\rangle$ is obtained from $|\Phi\rangle$ by
**exchanging all $\alpha$ and $\beta$ MOs**.

**Why swapping projects onto the singlet:** The triplet component ($S=1$, $M_S=0$)
of $|\Phi\rangle$ changes sign under $\alpha\leftrightarrow\beta$ exchange,
while the singlet component does not.
Therefore $|\Phi\rangle + |\tilde\Phi\rangle$ cancels the triplet
and doubles the singlet contribution.

This two-determinant form is the **first-order Lowdin projection**, exact when
only the $M_S=0$ triplet contaminates the wavefunction
(Sugisaki 2019, Section 2.2).

### 8.3 Normalisation Constant

Defining $S_{\rm swap} \equiv \langle\Phi|\tilde\Phi\rangle$ and using
$\langle\Phi|\Phi\rangle = \langle\tilde\Phi|\tilde\Phi\rangle = 1$:

$$\langle\Psi_{\rm PUHF}|\Psi_{\rm PUHF}\rangle
= \mathcal{N}^2(2 + 2\,S_{\rm swap}) = 1,
\qquad
\boxed{\mathcal{N} = \frac{1}{\sqrt{2 + 2\,S_{\rm swap}}}}.$$

### 8.4 Computing $S_{\rm swap}$

The overlap between two Slater determinants factorises by spin sector:

$$\langle\Phi_1|\Phi_2\rangle
= \det\bigl(\mathbf{S}^\alpha\bigr)\cdot\det\bigl(\mathbf{S}^\beta\bigr),$$

where $\mathbf{S}^\sigma_{ij} = \langle\psi^\sigma_i|\psi^\sigma_j\rangle
= (\mathbf{C}^{\sigma,{\rm occ}}_1)^\top \mathbf{S}_{\rm AO}\,\mathbf{C}^{\sigma,{\rm occ}}_2$.
For $S_{\rm swap}$: set $\mathbf{C}^\alpha_2 = \mathbf{C}^\beta$ and
$\mathbf{C}^\beta_2 = \mathbf{C}^\alpha$.


In [ ]:
# ---------------------------------------------------------------
# Step 8: Swap overlap and PUHF normalisation
# ---------------------------------------------------------------

def det_overlap_two_dets(mol, Ca1, Cb1, Ca2, Cb2):
    """
    <Phi_1 | Phi_2> = det(S_alpha) * det(S_beta)
    S_alpha = Ca1_occ^T @ S_AO @ Ca2_occ
    S_beta  = Cb1_occ^T @ S_AO @ Cb2_occ
    """
    S_ao          = mol.intor('int1e_ovlp')
    nalpha, nbeta = mol.nelec
    S_a = Ca1[:, :nalpha].T @ S_ao @ Ca2[:, :nalpha]
    S_b = Cb1[:, :nbeta].T  @ S_ao @ Cb2[:, :nbeta]
    return np.linalg.det(S_a) * np.linalg.det(S_b)


# S_swap = <Phi | Phi_tilde>  (Ca2=Cb, Cb2=Ca: the swap)
S_swap = det_overlap_two_dets(mol, Ca_uhf, Cb_uhf, Cb_uhf, Ca_uhf)
S_swap = float(np.clip(np.real(S_swap), -1.0, 1.0))
norm   = np.sqrt(2.0 + 2.0*S_swap)

print(f'S_swap = <Phi|Phi_tilde>     = {S_swap:.8f}')
print(f'Norm = sqrt(2 + 2*S_swap)    = {norm:.8f}')
print(f'Normalisation constant N      = {1.0/norm:.8f}')
if norm < 1e-12:
    print('  WARNING: norm < 1e-12 -- PUHF state ill-defined.')
    print('  The UHF may have collapsed to RHF (S_swap ~ -1).')


---
## Section 9 -- PUHF-CASSCF Overlap (Main Result)

### Complete Formula

Combining Sections 6-8 (Sugisaki 2019, Eqs. 6-7):

$$\boxed{
\langle\Psi_{\rm CAS}|\Psi_{\rm PUHF}\rangle
= \frac{\langle\Psi_{\rm CAS}|\Phi\rangle + \langle\Psi_{\rm CAS}|\tilde\Phi\rangle}
       {\sqrt{2 + 2\,S_{\rm swap}}}
}$$

An absolute value is taken for the physical quantity because the global
wavefunction phase is arbitrary.

The **QPEA success probability** is:

$$P_{\rm QPEA} = |\langle\Psi_{\rm CAS}|\Psi_{\rm PUHF}\rangle|^2.$$

| $|\langle\cdot\rangle|^2$ | QPEA efficiency |
|---|---|
| $\geq 0.9$ | Efficient: few repetitions needed |
| $0.5$-$0.9$ | Moderate |
| $< 0.5$ | Inefficient: consider a richer ansatz |


In [ ]:
# ---------------------------------------------------------------
# Step 9: PUHF-CASSCF overlap (main result)
# ---------------------------------------------------------------

if norm < 1e-12:
    O_puhf = 0.0
    print('PUHF overlap = 0 (norm ill-defined; see Step 8 warning).')
else:
    O_puhf = abs((O_phi + O_phi_tilde) / norm)

    print('PUHF-CASSCF overlap:')
    print(f'  <CAS|Phi>           = {O_phi:+.8f}')
    print(f'  <CAS|Phi_tilde>     = {O_phi_tilde:+.8f}')
    print(f'  Sum (numerator)     = {O_phi+O_phi_tilde:+.8f}')
    print(f'  Norm (denominator)  = {norm:.8f}')
    print(f'  |<CAS|PUHF>|        = {O_puhf:.8f}')
    print(f'  |<CAS|PUHF>|^2      = {O_puhf**2:.8f}  (QPEA success probability)')
    prob = O_puhf**2
    label = 'EFFICIENT (>=90%)' if prob>=0.9 else ('MODERATE (50-90%)' if prob>=0.5 else 'INEFFICIENT (<50%)')
    print(f'  QPEA efficiency:    {label}')


---
## Section 10 -- Summary and Physical Interpretation

### 10.1 Results Summary


In [ ]:
print('='*68)
print('SUGISAKI PUHF OVERLAP -- COMPLETE RESULTS')
print('='*68)
print(f'Molecule:       {XYZ_FILE}')
print(f'Basis:          {BASIS}')
print(f'Active space:   CAS({N_CAS_EL},{N_CAS_ORB})')

print('\n--- Energies ---')
print(f'  E(RHF)   = {E_rhf:.10f} Ha')
print(f'  E(UHF)   = {mf_uhf.e_tot:.10f} Ha')
print(f'  E(CAS)   = {mc.e_tot:.10f} Ha')

s2_val = mf_uhf.spin_square()[0]
print('\n--- UHF diagnostics ---')
print(f'  <S2>(UHF)        = {s2_val:.6f}  (ideal singlet: 0.000)')
print(f'  n_HONO           = {occ[n_occ-1]:.6f}  (ideal doubly-occ: 2.000)')
print(f'  n_LUNO           = {occ[n_occ]:.6f}  (ideal virtual: 0.000)')
print(f'  y0(UHF)          = {y_uhf[0]:.6f}  (diradical char., no spin proj.)')
print(f'  y0(PUHF, Eq. 4)  = {y_puhf[0]:.6f}  (spin-projected diradical char.)')

print('\n--- Wavefunction overlaps ---')
print(f"  {'Wavefunction':>12}  {'|overlap|':>12}  {'|overlap|^2':>14}")
print('  '+'-'*42)
print(f"  {'RHF':>12}  {abs(O_rhf):>12.8f}  {abs(O_rhf)**2:>14.8f}")
print(f"  {'PUHF':>12}  {O_puhf:>12.8f}  {O_puhf**2:>14.8f}")
if abs(O_rhf) > 1e-8:
    r = O_puhf / abs(O_rhf)
    print(f'\n  PUHF/RHF improvement: {r:.4f}x (overlap),  {r**2:.4f}x (probability)')
print('\n  S_swap =', f'{S_swap:.8f}',  '  N =', f'{1.0/norm:.8f}')
print('\n'+'='*68)


### 10.2 Physical Interpretation

#### RHF overlap and topological inaccessibility

The RHF wavefunction $|\Phi_{\rm RHF}\rangle$ is a closed-shell singlet.
For systems well described by a single configuration (aromatic TSs, reactants),
$n_{\rm HONO}\approx 2$ and $|\langle\Psi_{\rm CAS}|\Phi_{\rm RHF}\rangle|^2$
is large.

For antiaromatic or strongly diradicaloid systems, the HOMO-LUMO degeneracy
locus $\Sigma$ separates the closed-shell and open-shell singlet sectors
(UEIT and TOPCT of the main paper).  Symmetry-preserving unitary evolution
from a RHF initial state **cannot** transfer amplitude across $\Sigma$.
Therefore $|\langle\Psi_{\rm CAS}|\Phi_{\rm RHF}\rangle|^2$ is small when
the target state has significant diradicaloid character.

The LUNO occupation $n_{\rm LUNO}$ is a qualitative indicator of
proximity to $\Sigma$.

#### PUHF overlap and the role of symmetry breaking

The broken-symmetry UHF wavefunction partially captures static correlation
by mixing HOMO and LUMO.  Spin projection restores $S=0$ symmetry and
yields a higher overlap with the open-shell singlet CAS reference when
$n_{\rm LUNO} > 2/3$, as predicted by Sugisaki Eq. 4.

#### Connection to UEIT/TOPCT (main paper)

| System | $n_{\rm LUNO}$ | Sector | RHF overlap | PUHF advantage |
|---|---|---|---|---|
| Aromatic TS | ~0.1 | $\mathcal{C}_+$ | High (~0.91) | Minimal |
| Antiaromatic TS | ~0.5-0.7 | near/across $\Sigma$ | Low (~0.72) | Significant |
| Stabilised AA TS | ~0.5-0.6 | near $\Sigma$ | Moderate (~0.77) | Marginal |

Introduction of an aromatic $\pi$-linker stabilises the antiaromatic TS
energetically and reduces $n_{\rm LUNO}$, moving the system away from $\Sigma$.
Both RHF and PUHF overlaps improve, but the topological obstruction at $\Sigma$
itself persists regardless of the energetic cost of crossing it.
This distinction between energetic accessibility and topological accessibility
is the central message of the main paper.
